# 指标自带回测：`ind.signal_backtest()`

指标不只是画线：在 `step()` 里写买卖逻辑，就能用 `signal_backtest()` **一行回测**，拿到收益曲线与统计表。

**本次演示**

- `BtIndicator.step()` 里写信号（`self.long()` / `self.short()`）
- `signal_backtest()` 返回统计 / 曲线对象
- 同一指标也能 `backtest()` 走完整回测引擎

**前置条件**：`pip install minibt`（本页只用**内置数据**，
不需要联网 / 账号，也不会下单）。终端若为 GBK，先设 `PYTHONIOENCODING=utf-8`。

**对应文档**：[1.27minibt_indicator_backtest.md](../minibt_docs/docs/minibt_basic/1.27minibt_indicator_backtest.md)


## 1. 导入与内置数据


In [ ]:
from minibt import *

kline=LocalDatas.test.kline
kline.head()

## 2. 自定义指标（含 step 逻辑）


In [ ]:
class CCI(BtIndicator):
    """https://www.shinnytech.com/articles/trading-strategy/mean-reversion/cci-strategy"""
    isplot = dict(long_signal=False, short_signal=False)
    params = dict(CCI_PERIOD=10, CCI_UPPER=100, CCI_LOWER=-100)

    def next(self):
        CCI_PERIOD = self.params.CCI_PERIOD  # CCI计算周期
        CCI_UPPER = self.params.CCI_UPPER  # CCI上轨
        CCI_LOWER = self.params.CCI_LOWER  # CCI下轨
        cci = self.close.cci(CCI_PERIOD)
        size = cci.size
        cci_state = np.ones(size)#self.ones
        for i in range(CCI_PERIOD+1, size):
            cci_state[i] = cci[i] > CCI_UPPER and 1. or (
                cci[i] < CCI_LOWER and -1. or cci[i-1])
        cci_state = IndSeries(cci_state)
        long_signal = cci_state.cross_up(0.)
        short_signal = cci_state.cross_down(0.)
        return cci, long_signal, short_signal

cci=CCI(kline)
cci.tail()

## 3. signal_backtest()：一行回测


In [ ]:
sbr=cci.signal_backtest()

## 小结

- `step()` 是"逐根决策"的地方，写法与策略 `next()` 一致；
- `signal_backtest()` 用 Cython 加速，比完整回测快很多。

## 下一步

- 想把结论画进桌面端：见 [1.33 Gui.Desk](../minibt_docs/docs/minibt_basic/1.33minibt_desk_gui.md)；
- 想找其它例子：看 [`tutorials/README.md`](../README.md) 的目录表；
- 有疑问 / 想补充：[提 Issue](https://github.com/MiniBtMaster/minibt/issues)。
